# 📝 Week 5: Perbandingan Metode Keyword Extraction

Notebook ini membandingkan tiga metode populer untuk ekstraksi kata kunci pada artikel MBG:
- **TF-IDF** (Term Frequency - Inverse Document Frequency)
- **RAKE** (Rapid Automatic Keyword Extraction)
- **TextRank** (Graph-based ranking)

## 🎯 Tujuan Pembelajaran:
1. Memahami prinsip kerja TF-IDF, RAKE, dan TextRank
2. Mengimplementasikan ketiga metode pada dataset artikel MBG
3. Membandingkan hasil ketiga metode secara kualitatif
4. Mengidentifikasi kelebihan dan kelemahan masing-masing metode
5. Mengekspor hasil perbandingan ke format Excel

## 📦 Instalasi Dependencies

In [ ]:
import warnings
warnings.filterwarnings('ignore')

%pip install -q rake-nltk sumy nltk scikit-learn pandas openpyxl

print("✅ Semua library berhasil diinstall!")

## 📚 Import Libraries

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from rake_nltk import Rake
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print("✅ Import libraries berhasil!")

---

## 1️⃣ Load Dataset

In [ ]:
# Load dataset MBG
file_path = 'artikel_manual_celine.xlsx'
df_main = pd.read_excel(file_path, sheet_name='Main')

print("✅ Dataset berhasil dimuat!")
print(f"Total artikel: {len(df_main)}")
df_main[['Judul', 'Konten']].head(3)

---

## 2️⃣ Metode 1 — TF-IDF

TF-IDF mengukur kepentingan kata berdasarkan frekuensi di dokumen (TF) dan kelangkaan di seluruh korpus (IDF).

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\frac{N}{\text{DF}(t)}$$

**Keunggulan:** Cepat, deterministik, bekerja baik di level korpus.
**Kelemahan:** Tidak menangkap frasa multi-kata, mengabaikan urutan kata.

### 2.1 Setup TF-IDF

In [ ]:
# Setup TF-IDF
stop_words = stopwords.words('indonesian')

vectorizer = TfidfVectorizer(max_features=50, stop_words=stop_words)
X = vectorizer.fit_transform(df_main['Konten'].fillna(''))
tfidf_feature_names = vectorizer.get_feature_names_out()

def extract_tfidf_keywords(doc_index, top_n=5):
    """Ekstrak top-N kata kunci TF-IDF untuk satu dokumen."""
    row = X[doc_index].toarray().flatten()
    top_indices = row.argsort()[::-1][:top_n]
    return [tfidf_feature_names[i] for i in top_indices if row[i] > 0]

print("✅ TF-IDF vectorizer siap!")
print(f"Vocabulary size: {len(tfidf_feature_names)} kata")

In [ ]:
# Preview hasil TF-IDF untuk dokumen pertama
kw_tfidf_sample = extract_tfidf_keywords(0, top_n=10)
print("📌 Top 10 kata kunci TF-IDF (dokumen 1):")
print(kw_tfidf_sample)

---

## 3️⃣ Metode 2 — RAKE

RAKE (Rapid Automatic Keyword Extraction) mengidentifikasi kata kunci berdasarkan **frekuensi kata** dan **co-occurrence** — frasa dibatasi oleh stopwords dan tanda baca.

**Keunggulan:** Menangkap frasa multi-kata, tidak membutuhkan training.
**Kelemahan:** Sensitif terhadap daftar stopwords, bisa menghasilkan frasa panjang.

### 3.1 Setup RAKE

In [ ]:
# Setup RAKE
r = Rake(stopwords=stop_words)

def extract_rake_keywords(text, top_n=5):
    """Ekstrak top-N frasa kunci RAKE."""
    if pd.isna(text):
        return []
    r.extract_keywords_from_text(text)
    return [phrase for score, phrase in r.get_ranked_phrases_with_scores()[:top_n]]

print("✅ RAKE extractor siap!")

In [ ]:
# Preview hasil RAKE untuk dokumen pertama
kw_rake_sample = extract_rake_keywords(df_main['Konten'].iloc[0], top_n=10)
print("📌 Top 10 frasa kunci RAKE (dokumen 1):")
for i, phrase in enumerate(kw_rake_sample, 1):
    print(f"  {i}. {phrase}")

---

## 4️⃣ Metode 3 — TextRank

TextRank menggunakan **algoritma graf** (mirip PageRank) untuk menentukan kalimat atau kata mana yang paling penting berdasarkan hubungan antar kata/kalimat.

**Keunggulan:** Menghasilkan kalimat ringkasan yang informatif, mempertimbangkan konteks.
**Kelemahan:** Output berupa kalimat (bukan kata), lebih lambat.

### 4.1 Setup TextRank

In [ ]:
# Setup TextRank
summarizer = TextRankSummarizer()

def extract_textrank_keywords(text, top_n=3):
    """Ekstrak top-N kalimat penting menggunakan TextRank."""
    if pd.isna(text) or not isinstance(text, str):
        return []
    try:
        parser = PlaintextParser.from_string(text, Tokenizer('indonesian'))
    except LookupError:
        parser = PlaintextParser.from_string(text, Tokenizer('english'))
    summary = summarizer(parser.document, top_n)
    return [str(sentence) for sentence in summary]

print("✅ TextRank summarizer siap!")

In [ ]:
# Preview hasil TextRank untuk dokumen pertama
kw_textrank_sample = extract_textrank_keywords(df_main['Konten'].iloc[0], top_n=3)
print("📌 Top 3 kalimat penting TextRank (dokumen 1):")
for i, sent in enumerate(kw_textrank_sample, 1):
    print(f"  {i}. {sent[:120]}...")

---

## 5️⃣ Perbandingan Ketiga Metode pada Seluruh Dataset

In [ ]:
# Terapkan ketiga metode ke semua dokumen
comparison_results = []
tfidf_results = []
rake_results = []
textrank_results = []

for idx in range(len(df_main)):
    text = df_main.loc[idx, 'Konten']
    title = df_main.loc[idx, 'Judul']

    tfidf_kw = extract_tfidf_keywords(idx, top_n=5)
    rake_kw = extract_rake_keywords(text, top_n=5)
    textrank_kw = extract_textrank_keywords(text, top_n=3)

    comparison_results.append({
        'doc_id': idx,
        'title': title,
        'TF-IDF': tfidf_kw,
        'RAKE': rake_kw,
        'TextRank': textrank_kw
    })

    for kw in tfidf_kw:
        tfidf_results.append({'doc_id': idx, 'title': title, 'keyword': kw})
    for kw in rake_kw:
        rake_results.append({'doc_id': idx, 'title': title, 'keyword': kw})
    for kw in textrank_kw:
        textrank_results.append({'doc_id': idx, 'title': title, 'keyword': kw})

df_comparison = pd.DataFrame(comparison_results)
df_tfidf = pd.DataFrame(tfidf_results)
df_rake = pd.DataFrame(rake_results)
df_textrank = pd.DataFrame(textrank_results)

print("✅ Semua metode selesai diterapkan!")
print(f"  TF-IDF  : {len(df_tfidf)} keyword entries")
print(f"  RAKE    : {len(df_rake)} keyword entries")
print(f"  TextRank: {len(df_textrank)} keyword entries")

In [ ]:
# Preview tabel perbandingan
print("📊 Tabel Perbandingan Metode (5 dokumen pertama):")
df_comparison[['title', 'TF-IDF', 'RAKE', 'TextRank']].head(5)

---

## 6️⃣ Analisis Hasil Perbandingan

### 6.1 Kata Kunci TF-IDF Paling Sering Muncul

In [ ]:
# Top kata kunci TF-IDF
top_tfidf = df_tfidf['keyword'].value_counts().head(15).reset_index()
top_tfidf.columns = ['keyword', 'frequency']
print("🔑 Top 15 Kata Kunci TF-IDF:")
print(top_tfidf.to_string(index=False))

### 6.2 Frasa RAKE Paling Sering Muncul

In [ ]:
# Top frasa RAKE
top_rake = df_rake['keyword'].value_counts().head(15).reset_index()
top_rake.columns = ['keyword', 'frequency']
print("🔑 Top 15 Frasa RAKE:")
print(top_rake.to_string(index=False))

### 6.3 Perbandingan Karakteristik Metode

| Aspek | TF-IDF | RAKE | TextRank |
|---|---|---|---|
| Output | Kata tunggal/bigram | Frasa multi-kata | Kalimat lengkap |
| Pendekatan | Statistik | Statistik | Graph-based |
| Training | Tidak perlu | Tidak perlu | Tidak perlu |
| Konteks | Level korpus | Level dokumen | Level dokumen |
| Kecepatan | Sangat cepat | Cepat | Lebih lambat |
| Kelebihan | Komparasi antar dokumen | Menangkap frasa | Output narratif |
| Kelemahan | Kata tunggal saja | Bergantung stopwords | Output terlalu panjang |

---

## 7️⃣ Export Hasil ke Excel

In [ ]:
# Export semua hasil ke Excel dengan 4 sheet
output_file = 'mbg_keywords_comparison.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_tfidf.to_excel(writer, sheet_name='TF-IDF', index=False)
    df_rake.to_excel(writer, sheet_name='RAKE', index=False)
    df_textrank.to_excel(writer, sheet_name='TextRank', index=False)
    df_comparison.to_excel(writer, sheet_name='Comparison', index=False)

print(f"✅ Semua hasil diekspor ke '{output_file}'")
print("   Sheet: TF-IDF | RAKE | TextRank | Comparison")

---

## 📝 Kesimpulan

- 🎯 **TF-IDF** menonjolkan kata-kata bermuatan kebijakan & angka seperti *program*, *mbg*, *gizi*, *pemerintah* — cocok untuk analisis level korpus.
- 📊 **RAKE** mengidentifikasi frasa-frasa yang mencerminkan kekhawatiran sosial & kesehatan seperti *"keracunan makanan bergizi"*, *"siswa sakit"* — cocok untuk analisis wacana.
- 🔧 **TextRank** menghasilkan kalimat naratif yang merangkum isi artikel secara holistik — cocok untuk ringkasan otomatis.
- 📂 Ketiga metode saling melengkapi; kombinasi ketiganya memberikan gambaran paling komprehensif tentang framing berita MBG.

---

## 📚 Referensi

- Rose, S., et al. (2010). Automatic keyword extraction from individual documents. *Text Mining: Applications and Theory*.
- Mihalcea, R., & Tarau, P. (2004). TextRank: Bringing order into texts. *EMNLP*.
- Scikit-learn: [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- RAKE-NLTK: [https://github.com/csurfer/rake-nltk](https://github.com/csurfer/rake-nltk)
- Sumy: [https://github.com/miso-belica/sumy](https://github.com/miso-belica/sumy)

---

**Terima kasih! Happy Learning! 🎓**